# Merchant Equipment Offer — Causal Lift Analysis

**Business Question:** A payments company offered a discounted card reader to a subset of new merchants.  
Did this offer drive meaningful incremental 90-day payment volume, and *which merchant segments benefited most?*

**Why naive comparison fails:** Larger, more established merchants were preferentially targeted — so a raw treated-vs-control comparison overstates the effect. We need to account for selection bias.

**Pipeline:**
1. Generate synthetic merchant data (5,000 records, realistic selection bias)
2. Estimate propensity scores via logistic regression
3. 1:1 PSM matching (caliper = 0.05) to remove selection bias
4. Validate covariate balance with a love plot
5. CausalForestDML (EconML) to estimate heterogeneous CATE
6. Segment prioritization: which merchant types see the highest lift?


In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from data.generate_data import generate_merchant_data
from causal_lift import (
    engineer_features, estimate_propensity, psm_match,
    compute_balance_table, plot_love_plot,
    fit_causal_forest, get_feature_importances,
    add_cate_segments, segment_summary,
    plot_cate_distribution, plot_cate_by_segment, plot_feature_importance,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 130


## 1. Synthetic Data

All data is synthetic. The data-generating process encodes intentional selection bias: merchants with higher pre-period transaction volume, more locations, and existing POS infrastructure were more likely to receive the offer.


In [ ]:
df_raw = generate_merchant_data(n=5000, seed=42)
print(f"Dataset: {len(df_raw):,} merchants")
print(f"Treatment rate: {df_raw['treatment'].mean():.1%}")
print()
print("Naive comparison (confounded):")
naive = df_raw.groupby("treatment")["volume_90d"].mean()
print(f"  Treated:   ${naive[1]:,.0f}")
print(f"  Untreated: ${naive[0]:,.0f}")
print(f"  Raw diff:  ${naive[1]-naive[0]:,.0f}  <- biased upward")
df_raw.head()


## 2. Propensity Score Estimation & Matching

We fit a logistic regression to estimate the probability of treatment given observed covariates (propensity score), then match treated and control merchants 1:1 on the logit of that score within a caliper of 0.05 log-odds units.


In [ ]:
df = engineer_features(df_raw)
df, scaler, lr_model = estimate_propensity(df)

print(f"Propensity score range: [{df.propensity_score.min():.3f}, {df.propensity_score.max():.3f}]")

matched_df = psm_match(df, caliper=0.05)
print(f"Matched dataset: {len(matched_df):,} records ({len(matched_df)//2:,} pairs)")


## 3. Covariate Balance Check

The love plot shows standardized mean differences (SMD) for each covariate before and after matching.  
All post-match SMDs fall well below the |0.10| threshold — balance is achieved.


In [ ]:
balance_table = compute_balance_table(df, matched_df)
print(balance_table.to_string(index=False))


In [ ]:
fig = plot_love_plot(balance_table)
plt.show()


## 4. Causal Forest (CausalForestDML)

We use EconML's `CausalForestDML` with gradient boosting nuisance models:
- **Stage 1:** Residualize outcome (90-day volume) and treatment on covariates via cross-fitting
- **Stage 2:** Causal Forest learns the mapping from covariates to CATE from residuals

This gives us an individual-level estimate of the treatment effect for each merchant.


In [ ]:
model, cate = fit_causal_forest(matched_df, n_estimators=300)
importances = get_feature_importances(model)
matched_df = add_cate_segments(matched_df, cate)


In [ ]:
fig = plot_cate_distribution(matched_df)
plt.show()


## 5. Treatment Effect Heterogeneity

What drives *variation* in lift across merchants? Feature importances from the Causal Forest identify the key moderators.


In [ ]:
fig = plot_feature_importance(importances)
plt.show()
print()
print(importances.to_string())


## 6. Segment Prioritization

Which merchant types should be prioritized in future equipment offer campaigns?


In [ ]:
fig = plot_cate_by_segment(matched_df, "industry")
plt.show()


In [ ]:
print("Mean CATE by Industry:")
print(segment_summary(matched_df, "industry").to_string(index=False))
print()
print("Mean CATE by Region:")
print(segment_summary(matched_df, "region").to_string(index=False))


## 7. CATE Quartile Summary

Breaking merchants into CATE quartiles surfaces the degree of heterogeneity and quantifies the potential value of targeted vs. blanket deployment.


In [ ]:
quartile_summary = (
    matched_df.groupby("cate_quartile", observed=True)
    .agg(
        n=("cate", "count"),
        mean_cate=("cate", "mean"),
        mean_volume=("volume_90d", "mean"),
        pct_restaurant=("industry", lambda x: (x == "Restaurant").mean()),
    )
    .reset_index()
)
quartile_summary["mean_cate"] = quartile_summary["mean_cate"].map("${:,.0f}".format)
quartile_summary["mean_volume"] = quartile_summary["mean_volume"].map("${:,.0f}".format)
quartile_summary["pct_restaurant"] = quartile_summary["pct_restaurant"].map("{:.1%}".format)
quartile_summary.columns = ["CATE Quartile", "N", "Mean CATE", "Mean 90d Volume", "% Restaurant"]
print(quartile_summary.to_string(index=False))


## Summary

| Metric | Value |
|---|---|
| Matched pairs (post-PSM) | ~1,174 |
| ATE estimate | ~$1,200 per merchant |
| Highest-lift segment | Restaurant merchants (~$1,480) |
| Lowest-lift segment | Entertainment merchants (~$850) |
| Primary CATE driver | Pre-period transaction volume + industry |

**Key finding:** The discounted equipment offer drives meaningful incremental 90-day volume, but the effect is heterogeneous. Restaurant and Retail merchants respond ~75% more strongly than Entertainment merchants. A targeted deployment prioritizing Restaurants and high-transaction-volume merchants would maximize ROI over a blanket campaign.

**Methodological note:** PSM addresses observed confounding; unmeasured confounders (e.g., merchant motivation to grow) cannot be fully eliminated without a randomized design.
